In [ ]:
import sys
from pathlib import Path

HICOLA_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'hicola_paths.py').exists())
sys.path.insert(0, str(HICOLA_ROOT / 'scripts'))
import hicola_paths as hp
import analysis as an

import numpy as np
import matplotlib.pyplot as plt

# nDGP signal in the HOD galaxy power spectrum

$S_X(k) = X_\mathrm{nDGP}(k) / X_\mathrm{GR}(k) - 1$ for the redshift-space multipoles $X = P_0, P_2$
of the HOD mocks made by `jobs/hod_mock.sh` (measured as in the EMC/acm pipeline). The fine k-bins of
the measurement are rebinned to width `dk`, weighting by the number of modes.

Both mocks use the same HOD parameters; their galaxy number densities and biases can differ
(see the printed densities), which enters $S_X$ together with the effect of gravity.

In [ ]:
target = 'Abacus_Small'
redshift = 0.5
realization = 's1234'
name = 'LRG_seed42'          # NAME in jobs/hod_mock.sh
gr_model = 'LCDM'
ndgp_model = 'nDGP_N1.2'
ells = (0, 2)                # X = P_ell
dk = 0.01                    # rebinned k-bin width [h/Mpc]
k_max = 0.5                  # largest k shown [h/Mpc]

## Load

In [ ]:
spectra = {}
for model in (gr_model, ndgp_model):
    spectra[model] = an.load_hod_spectrum(target, model, realization, redshift, name, dk)
    meta_file = Path(str(hp.hicola_hod_spectrum_file(target, model, realization, redshift, name)) + '.json')
    if meta_file.exists():
        import json
        meta = json.loads(meta_file.read_text())
        print(f"{model}: {meta['n_galaxies']:,} galaxies, n = {meta['number_density_h3_Mpc3']:.3e} (h/Mpc)^3")

k, P_gr = spectra[gr_model]
k_ndgp, P_ndgp = spectra[ndgp_model]
assert np.allclose(k, k_ndgp), 'the two measurements have different k-bins'
keep = k <= k_max
S = {ell: P_ndgp[ell][keep] / P_gr[ell][keep] - 1 for ell in ells}
k = k[keep]

## $S_X(k)$

In [ ]:
fig, axes = plt.subplots(1, len(ells), figsize=(3.4 * len(ells), 3.0), dpi=200, constrained_layout=True)
for ax, ell in zip(np.atleast_1d(axes), ells):
    ax.plot(k, S[ell], color='#c47f2b', lw=1.4)
    ax.axhline(0, color='0.4', lw=0.8, ls=':')
    ax.set_xscale('log')
    ax.set_xlabel(r'$k\,[h\,\mathrm{Mpc}^{-1}]$')
    ax.set_ylabel(rf'$S_{{P_{ell}}}(k) = P_{ell}^\mathrm{{nDGP}} / P_{ell}^\mathrm{{GR}} - 1$')
    ax.set_title(rf'$P_{ell}$, {target}, z = {redshift}', fontsize=9)
    ax.minorticks_on()
    ax.grid(True, which='major', ls=':', color='0.82', lw=0.45)
    ax.tick_params(which='both', direction='in', top=True, right=True)
plt.show()